# AGCRN Single-Notebook Runner for METR-LA
This notebook clones AGCRN, patches it for METR-LA, creates config files, trains, and evaluates.

In [1]:
!git clone https://github.com/LeiBAI/AGCRN.git


fatal: destination path 'AGCRN' already exists and is not an empty directory.


In [2]:
import pandas as pd, numpy as np
from pathlib import Path
df = pd.read_hdf('metr-la.h5')
print(df.shape)
Path('AGCRN/data/METRLA').mkdir(parents=True, exist_ok=True)


(34272, 207)


In [3]:
from pathlib import Path
loader = Path('AGCRN/lib/load_dataset.py')
txt = loader.read_text()
old = """    elif dataset == 'PEMSD8':\n        data_path = os.path.join('../data/PeMSD8/pems08.npz')\n        data = np.load(data_path)['data'][:, :, 0]  #onley the first dimension, traffic flow data\n    else:\n        raise ValueError"""
new = """    elif dataset == 'PEMSD8':\n        data_path = os.path.join('../data/PeMSD8/pems08.npz')\n        data = np.load(data_path)['data'][:, :, 0]\n    elif dataset == 'METRLA':\n        import pandas as pd\n        data = pd.read_hdf('../data/METRLA/metr-la.h5').values\n    else:\n        raise ValueError"""
loader.write_text(txt.replace(old,new))
print('patched')


patched


In [4]:
import shutil
shutil.copy('metr-la.h5','AGCRN/data/METRLA/metr-la.h5')


'AGCRN/data/METRLA/metr-la.h5'

In [5]:
conf = '''
[data]
num_nodes = 207
lag = 12
horizon = 12
val_ratio = 0.2
test_ratio = 0.2
tod = False
normalizer = std
column_wise = False
default_graph = True

[model]
input_dim = 1
output_dim = 1
embed_dim = 10
rnn_units = 64
num_layers = 2
cheb_order = 2

[train]
loss_func = mae
seed = 10
batch_size = 64
epochs = 100
lr_init = 0.003
lr_decay = False
lr_decay_rate = 0.3
lr_decay_step = 5,20,40,70
early_stop = True
early_stop_patience = 15
grad_norm = False
max_grad_norm = 5
real_value = True

[test]
mae_thresh = None
mape_thresh = 0.

[log]
log_step = 20
plot = False
'''

In [6]:
from pathlib import Path
p=Path('AGCRN/model/Run.py')
txt=p.read_text()
txt=txt.replace("DATASET = 'PEMSD4'","DATASET = 'METRLA'")
p.write_text(txt)
print('Run.py patched')


Run.py patched


In [7]:
%cd AGCRN/model
!python Run.py


c:\Users\student\paperTest\AGCRN\model
c:\Users\student\paperTest\AGCRN
Read configuration file: ./METRLA_AGCRN.conf


Traceback (most recent call last):
  File "c:\Users\student\paperTest\AGCRN\model\Run.py", line 100, in <module>
    args.add_argument('--log_step', default=config['log']['log_step'], type=int)
                                            ~~~~~~^^^^^^^
  File "C:\Users\student\AppData\Local\Programs\Python\Python311\Lib\configparser.py", line 978, in __getitem__
    raise KeyError(key)
KeyError: 'log'
